# ChemBreak Task Bank Generator V7.0

This notebook creates harmful chemistry **target goals** for authorized LLM safety and jailbreak evaluation. It never asks a model to answer those goals.

- Qwen and Ministral each generate one fresh candidate per assignment.
- Deterministic validation checks all four candidate fields and gives exact, category-specific feedback to retries.
- Phi-4 independently qualifies both candidates, applies defect flags, and selects a winner blindly.
- `test`: 1 assignment and up to 1 selected goal.
- `pilot`: 29 assignments, covering every production plan and all 12 request-form families.
- `production`: 550 initial assignments, including 50 reserve assignments, for exactly 500 selected goals.

Only one checkpoint is loaded on the GPU at a time. Every candidate and judgment is saved immediately.


## 1. Get the current ChemBreak code

The repository is cloned on the first run and fast-forwarded on later runs. The V7 folder name is explicit so a repository layout mistake is reported immediately.


In [ ]:
from pathlib import Path
import subprocess
import sys

REPO_DIR = Path("/content/ChemBreak")
REPO_URL = "https://github.com/Jollychuks/ChemBreak.git"
PROJECT_SUBDIR = "ChemBreak_TaskBank_Generator_v7"

if (REPO_DIR / ".git").exists():
    subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--ff-only"], check=True)
elif REPO_DIR.exists():
    raise RuntimeError(
        f"{REPO_DIR} exists but is not a Git repository. Rename or remove that "
        "Colab folder, then rerun this cell."
    )
else:
    subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)

PROJECT_DIR = REPO_DIR / PROJECT_SUBDIR
PIPELINE = PROJECT_DIR / "chembreak_pipeline.py"
REQUIREMENTS = PROJECT_DIR / "requirements_colab.txt"

if not PIPELINE.is_file():
    available = sorted(path.name for path in REPO_DIR.iterdir() if path.is_dir())
    raise FileNotFoundError(
        f"Expected {PIPELINE}, but it was not found. "
        f"Top-level repository folders: {available}"
    )

print(f"Ready: {PROJECT_DIR}")
print(f"Python: {sys.executable}")


## 2. Install the runtime dependencies

Colab's installed PyTorch is kept. This cell adds the loaders needed by Qwen, Ministral, and Phi-4.


In [ ]:
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-r", str(REQUIREMENTS)],
    check=True,
)
print("Dependencies installed.")


## 3. Check the GPU, loaders, and model access

The checkpoints are public. A Hugging Face token is optional and can improve rate limits. If you saved an `HF_TOKEN` in Colab Secrets, this cell uses it without displaying it. No login widget is required.


In [ ]:
import os

os.environ["HF_HOME"] = "/content/hf_cache"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

try:
    from google.colab import userdata
    optional_token = userdata.get("HF_TOKEN")
except Exception:
    optional_token = None

if optional_token:
    os.environ["HF_TOKEN"] = optional_token
    print("Optional Hugging Face token found.")
else:
    print("No Hugging Face token found. Public downloads will still be attempted.")

PREFLIGHT_DIR = Path("/content/chembreak_v7_preflight")
subprocess.run(
    [
        sys.executable,
        "-u",
        str(PIPELINE),
        "--project-dir",
        str(PROJECT_DIR),
        "--output-dir",
        str(PREFLIGHT_DIR),
        "--run-type",
        "test",
        "--stage",
        "preflight",
    ],
    check=True,
    env={**os.environ, "PYTHONUNBUFFERED": "1"},
)


## 4. Choose the run

Start with `test`, then run `pilot`. Production saves resumable checkpoints to Google Drive. The explicit production quota is fixed at 500 final goals.


In [ ]:
RUN_TYPE = "test"  # "test", "pilot", or "production"
PRODUCTION_TARGET = 500
SESSION_HOURS = 9.5

if RUN_TYPE not in {"test", "pilot", "production"}:
    raise ValueError("RUN_TYPE must be test, pilot, or production.")

if RUN_TYPE == "production":
    from google.colab import drive
    drive.mount("/content/drive")
    OUTPUT_DIR = Path("/content/drive/MyDrive/ChemBreak_V7_Results/production")
else:
    OUTPUT_DIR = Path("/content/chembreak_v7_results") / RUN_TYPE

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print(f"Run type: {RUN_TYPE}")
print(f"Output directory: {OUTPUT_DIR}")


## 5. Live stage runner

Run this helper once. Every stage below streams progress and errors line by line. Completed rows are resumed rather than repeated.


In [ ]:
def run_stage(stage):
    command = [
        sys.executable,
        "-u",
        str(PIPELINE),
        "--project-dir",
        str(PROJECT_DIR),
        "--output-dir",
        str(OUTPUT_DIR),
        "--run-type",
        RUN_TYPE,
        "--stage",
        stage,
    ]
    if RUN_TYPE == "production":
        command.extend(["--target", str(PRODUCTION_TARGET)])
        if stage == "all":
            command.extend(["--session-hours", str(SESSION_HOURS)])

    print(f"Starting V7 stage: {stage}\n", flush=True)
    process = subprocess.Popen(
        command,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
        env={**os.environ, "PYTHONUNBUFFERED": "1"},
    )
    assert process.stdout is not None
    for line in process.stdout:
        print(line, end="", flush=True)
    return_code = process.wait()
    if return_code != 0:
        raise subprocess.CalledProcessError(return_code, command)
    print(f"\nStage completed: {stage}", flush=True)


## 6. Run individual resumable stages

Use these cells when a long session must be split. The normal order is plan, Qwen, Ministral, validation, judge, finalize. A refill cell adds one grouped deficit batch; after it, rerun the two generator cells, validation, judge, and finalize.


In [ ]:
run_stage("plan")


In [ ]:
run_stage("generate_qwen")


In [ ]:
run_stage("generate_ministral")


In [ ]:
run_stage("validate")


In [ ]:
run_stage("judge")


In [ ]:
run_stage("finalize")


In [ ]:
run_stage("refill")


## 7. Or run the complete workflow

This single cell runs every stage in order and performs grouped production refills. Rerun it after a Colab reset to resume.


In [ ]:
run_stage("all")


## 8. Inspect the current task bank and reports

`harmbench_behaviors.csv` is the direct goal list. `final_task_bank.csv` contains the full ChemBreak metadata. The summary clearly labels a checkpoint as incomplete or complete.


In [ ]:
import json
import pandas as pd
from IPython.display import display

summary_path = OUTPUT_DIR / "run_summary.json"
if summary_path.is_file():
    summary = json.loads(summary_path.read_text(encoding="utf-8"))
    print(json.dumps(summary, indent=2))
else:
    summary = {}
    print("No run summary exists yet.")

for filename in (
    "quota_report.csv",
    "coverage_report.csv",
    "request_form_report.csv",
    "diversity_report.csv",
    "generation_failures.csv",
    "judge_defect_report.csv",
    "final_task_bank.csv",
    "harmbench_behaviors.csv",
):
    path = OUTPUT_DIR / filename
    if path.is_file():
        frame = pd.read_csv(path)
        print(f"\n{filename}: {len(frame):,} row(s)")
        display(frame.head(10))
    else:
        print(f"\n{filename}: not written yet")


## 9. Download the current results

The archive name contains `CHECKPOINT` until exactly the requested task count is complete. A complete production archive contains `COMPLETE_500_OF_500`.


In [ ]:
import shutil
from google.colab import files

result_files = [path for path in OUTPUT_DIR.rglob("*") if path.is_file()]
if not result_files:
    raise FileNotFoundError(f"No result files exist in {OUTPUT_DIR}.")

summary_path = OUTPUT_DIR / "run_summary.json"
if summary_path.is_file():
    summary = json.loads(summary_path.read_text(encoding="utf-8"))
    label = summary.get("completion_label", "CHECKPOINT_UNKNOWN")
else:
    label = "CHECKPOINT_NO_SUMMARY"

archive_base = Path("/content") / f"ChemBreak_V7_{RUN_TYPE}_{label}"
archive_path = Path(shutil.make_archive(str(archive_base), "zip", root_dir=str(OUTPUT_DIR)))
print(f"Created {archive_path.name} ({archive_path.stat().st_size:,} bytes)")
files.download(str(archive_path))
